# 04 - Iterators, Generators & Context Managers

Part of the Python, DSA & Git chapter. This notebook covers exactly how Python actually executes `for x in thing` and `with thing as x`, and why generators are the standard way ML pipelines stream large datasets without loading everything into memory at once.

Covers: the iterator protocol, writing generators with yield, lazy vs eager evaluation, itertools, the with statement and custom context managers, and resource-cleanup patterns. Practice exercises at the end.

# Part A - The Iterator Protocol

## __iter__ and __next__: what "iterable" actually means

An object is iterable if it has an __iter__ method that returns an iterator. An iterator is an object with a __next__ method that returns the next value each time it is called, and raises StopIteration when there are no more values. A for loop is syntactic sugar over calling iter() once and next() repeatedly, catching StopIteration to know when to stop.

In [1]:
class CountUp:
    '''A custom iterator: counts from start to end, inclusive.'''
    def __init__(self, start, end):
        self.current = start
        self.end = end

    def __iter__(self):
        return self          # an iterator is its own iterator

    def __next__(self):
        if self.current > self.end:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

for x in CountUp(1, 5):
    print(x)

# What the for loop above is actually doing, spelled out manually:
it = iter(CountUp(1, 5))    # calls __iter__
while True:
    try:
        value = next(it)     # calls __next__
    except StopIteration:
        break
    print("manual:", value)

1
2
3
4
5
manual: 1
manual: 2
manual: 3
manual: 4
manual: 5


## Iterable vs iterator: a distinction interviewers like to probe

A list is iterable but is not itself an iterator -- calling iter() on it returns a fresh iterator object each time, which is exactly why the same list can be looped over twice, or two independent loops can run over it at once, without interfering with each other. CountUp above breaks that: because it returns self from __iter__, exhausting one loop over a CountUp instance exhausts it for good.

In [2]:
nums = [1, 2, 3]
it1 = iter(nums)
it2 = iter(nums)
print(next(it1), next(it1))   # advancing it1 does not affect it2
print(next(it2))               # it2 is still at the start

c = CountUp(1, 3)
for x in c:
    print("first pass:", x)
for x in c:
    print("second pass:", x)   # nothing prints -- c is exhausted, it IS its own used-up iterator

print(list(c))                  # confirms it is empty now

1 2
1
first pass: 1
first pass: 2
first pass: 3
[]


# Part B - Generators

## yield: generator functions write the iterator protocol for you

A function containing yield is a generator function -- calling it does not run the body immediately, it returns a generator object that implements __iter__ and __next__ automatically. Each call to next() resumes execution right after the last yield, running until the next yield, or until the function ends, which raises StopIteration automatically.

In [3]:
def count_up(start, end):
    current = start
    while current <= end:
        yield current
        current += 1

gen = count_up(1, 5)
print(type(gen))           # <class 'generator'>
print(next(gen), next(gen))
for x in gen:               # resumes from where next() left off -- prints 3, 4, 5
    print("resumed:", x)

# Compare the class-based CountUp above to this generator -- same behavior, far less
# boilerplate. This is why yield is almost always preferred over hand-writing
# __iter__/__next__ unless something a plain generator cannot express is needed.

<class 'generator'>
1 2
resumed: 3
resumed: 4
resumed: 5


## Generator expressions vs list comprehensions: lazy vs eager

A list comprehension builds the entire list in memory immediately. A generator expression, same syntax with parentheses instead of brackets, produces values lazily, one at a time, on demand. For large datasets this difference is the entire reason ML data pipelines use generators: it allows streaming through data far larger than memory.

In [4]:
import sys

list_comp = [x**2 for x in range(100_000)]      # built ALL AT ONCE, right now
gen_expr = (x**2 for x in range(100_000))         # built LAZILY, nothing computed yet

print("list comprehension size:", sys.getsizeof(list_comp), "bytes")
print("generator expression size:", sys.getsizeof(gen_expr), "bytes  <- constant, regardless of range size")

# Proof the generator has not actually computed anything yet:
big_gen = (x**2 for x in range(10**12))    # a trillion elements -- instant, no error, no memory spike
print(next(big_gen), next(big_gen), next(big_gen))

list comprehension size: 800984 bytes
generator expression size: 200 bytes  <- constant, regardless of range size
0 1 4


## Lazy evaluation with infinite generators

Because a generator only computes a value when asked, it can represent an infinite sequence, something a list fundamentally cannot do. itertools.islice takes a finite slice from an infinite generator without ever trying to materialize the whole thing.

In [5]:
from itertools import islice

def natural_numbers():
    n = 1
    while True:            # runs forever -- fine, because nothing forces it to finish
        yield n
        n += 1

first_ten = list(islice(natural_numbers(), 10))
print(first_ten)

def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

print(list(islice(fibonacci(), 10)))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


## yield from: delegating to a sub-generator

yield from forwards every value produced by an inner iterable, without writing an explicit loop. It is commonly used to flatten nested generators or to compose several generators into one pipeline.

In [6]:
def inner(n):
    for i in range(n):
        yield i

def outer():
    yield "start"
    yield from inner(3)     # equivalent to: for x in inner(3): yield x
    yield "end"

print(list(outer()))

# A realistic use: flatten a nested structure, e.g. batches of batches
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)     # recursive delegation
        else:
            yield item

nested_batches = [[1, 2, [3, 4]], [5], [[6, 7], 8]]
print(list(flatten(nested_batches)))

['start', 0, 1, 2, 'end']
[1, 2, 3, 4, 5, 6, 7, 8]


## A realistic ML use case: streaming a large file without loading it into memory

This is the exact pattern behind reading massive training datasets, such as logs, JSONL corpora, or CSVs too large for RAM: open the file, yield one processed record at a time, and let the caller decide how much to consume.

In [7]:
import pathlib

# Create a synthetic "large" file to stream through
data_path = pathlib.Path("synthetic_dataset.txt")
with open(data_path, "w") as f:
    for i in range(50_000):
        f.write(f"row_{i},{i*2}\n")

def stream_rows(path):
    '''Yield one parsed row at a time -- the file is never fully loaded into memory.'''
    with open(path) as f:
        for line in f:                      # files are themselves iterators over their lines
            name, value = line.strip().split(",")
            yield name, int(value)

row_gen = stream_rows(data_path)
print(next(row_gen))
print(next(row_gen))

# Process only what is needed, e.g. the first 3 rows where value > 5, without ever
# holding all 50,000 rows in memory at once
filtered = (row for row in stream_rows(data_path) if row[1] > 5)
print(list(islice(filtered, 3)))

data_path.unlink()   # cleanup the synthetic file

('row_0', 0)
('row_1', 2)
[('row_3', 6), ('row_4', 8), ('row_5', 10)]


# Part C - itertools: the Standard Toolkit for Iterator Composition

## itertools functions worth knowing cold

chain concatenates iterables; islice slices lazily; groupby groups CONSECUTIVE equal keys, meaning it does not sort first; product/permutations/combinations cover combinatorics; count/cycle/repeat build infinite sequences.

In [8]:
from itertools import chain, islice, groupby, product, permutations, combinations, count, cycle, repeat

print(list(chain([1, 2], [3, 4], [5])))              # concatenate several iterables

print(list(islice(count(10, 2), 5)))                   # 10, 12, 14, ... take 5

# groupby groups CONSECUTIVE matching keys -- data usually needs sorting first
data = [("a", 1), ("a", 2), ("b", 3), ("a", 4)]         # note: "a" appears in two separate runs
grouped = [(k, list(v)) for k, v in groupby(data, key=lambda x: x[0])]
print("without sorting first, a appears twice:", grouped)

data_sorted = sorted(data, key=lambda x: x[0])
grouped_sorted = [(k, list(v)) for k, v in groupby(data_sorted, key=lambda x: x[0])]
print("after sorting first:", grouped_sorted)

print(list(product([0, 1], repeat=3)))                 # all 2^3 binary combinations
print(list(permutations([1, 2, 3], 2)))                 # ordered pairs
print(list(combinations([1, 2, 3], 2)))                  # unordered pairs

print(list(islice(cycle([1, 2, 3]), 7)))                 # repeats the sequence forever
print(list(repeat("x", 4)))                               # repeats one value n times

[1, 2, 3, 4, 5]
[10, 12, 14, 16, 18]
without sorting first, a appears twice: [('a', [('a', 1), ('a', 2)]), ('b', [('b', 3)]), ('a', [('a', 4)])]
after sorting first: [('a', [('a', 1), ('a', 2), ('a', 4)]), ('b', [('b', 3)])]
[(0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)]
[(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
[(1, 2), (1, 3), (2, 3)]
[1, 2, 3, 1, 2, 3, 1]
['x', 'x', 'x', 'x']


# Part D - Context Managers: with, __enter__, __exit__

## The with statement and the __enter__/__exit__ protocol

`with obj as x:` runs obj.__enter__(), whose return value is bound to x, executes the block, and GUARANTEES obj.__exit__() runs afterward, even if the block raises an exception. This guarantee is the entire reason with exists: it replaces error-prone manual try/finally cleanup.

In [9]:
import time

class Timer:
    '''A context manager that times the block it wraps.'''
    def __enter__(self):
        self.start = time.perf_counter()
        return self                     # this is what gets bound to `as t`

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.perf_counter() - self.start
        print(f"block took {self.elapsed*1000:.2f} ms")
        return False                    # False means: do not suppress any exception

with Timer() as t:
    total = sum(range(1_000_000))
print("computed total:", total)

# __exit__ runs even when the block raises -- proving the cleanup guarantee
try:
    with Timer():
        raise ValueError("something broke inside the block")
except ValueError as e:
    print("exception still propagated after cleanup ran:", e)

block took 11.49 ms
computed total: 499999500000
block took 0.00 ms
exception still propagated after cleanup ran: something broke inside the block


## contextlib.contextmanager: writing context managers with a generator instead of a class

For simple cases, a full class with __enter__/__exit__ is more boilerplate than needed. @contextmanager turns a generator function into a context manager: code before yield is __enter__, the yielded value is what `as x` binds to, and code after yield, especially inside a finally block, is __exit__.

In [10]:
from contextlib import contextmanager

@contextmanager
def timer_cm(label):
    start = time.perf_counter()
    try:
        yield             # the with-block runs here
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] took {elapsed*1000:.2f} ms")

with timer_cm("sum computation"):
    total = sum(range(1_000_000))

# The finally guarantees the timing print still happens even if the block raises
try:
    with timer_cm("failing block"):
        raise RuntimeError("boom")
except RuntimeError as e:
    print("propagated:", e)

[sum computation] took 11.33 ms
[failing block] took 0.00 ms
propagated: boom


## Real resource-cleanup patterns

This is the actual reason context managers exist in production code: files, database connections, network sockets, and GPU memory all need guaranteed cleanup, and with is how Python expresses that guarantee. Nesting several resources is common enough that contextlib.ExitStack exists specifically to manage a dynamic number of them.

In [11]:
# The standard, idiomatic file-handling pattern -- the file is guaranteed closed
# even if reading raises partway through.
with open("temp_example.txt", "w") as f:
    f.write("hello\nworld\n")

with open("temp_example.txt") as f:
    print(f.closed)          # False -- still open, inside the with block
    lines = f.readlines()
print(f.closed)               # True -- __exit__ closed it automatically on block exit
pathlib.Path("temp_example.txt").unlink()

# Simulating a resource that needs explicit cleanup, e.g. a DB connection or GPU buffer
class FakeDBConnection:
    def __enter__(self):
        print("opening connection")
        self.open = True
        return self

    def query(self, sql):
        if not self.open:
            raise RuntimeError("connection is closed")
        return f"result of: {sql}"

    def __exit__(self, exc_type, exc_value, traceback):
        print("closing connection")
        self.open = False
        return False

with FakeDBConnection() as conn:
    print(conn.query("SELECT * FROM experiments"))
print("connection open after block:", conn.open)

# ExitStack -- for managing a variable number of context managers at once
from contextlib import ExitStack

paths = ["temp_a.txt", "temp_b.txt", "temp_c.txt"]
for p in paths:
    pathlib.Path(p).write_text("data")

with ExitStack() as stack:
    files = [stack.enter_context(open(p)) for p in paths]   # all opened, ALL guaranteed closed
    print("all closed while inside the block?", [f.closed for f in files])
print("after ExitStack exits:", [f.closed for f in files])

for p in paths:
    pathlib.Path(p).unlink()

False
True
opening connection
result of: SELECT * FROM experiments
closing connection
connection open after block: False
all closed while inside the block? [False, False, False]
after ExitStack exits: [True, True, True]


## contextlib.suppress: a small, targeted convenience

Turns a specific try/except that only exists to ignore an exception into a single line.

In [12]:
from contextlib import suppress

with suppress(FileNotFoundError):
    pathlib.Path("does_not_exist.txt").unlink()   # would normally raise -- suppressed instead
print("execution continues normally after the suppressed block")

execution continues normally after the suppressed block


## Practice exercises

Implement each TODO, then run the check cell.

In [13]:
def chunked(iterable, size):
    # Yield successive lists of length `size` from iterable (the last chunk may be shorter).
    # e.g. chunked([1,2,3,4,5,6,7], 3) -> [1,2,3], [4,5,6], [7]
    # TODO: implement as a generator using yield
    raise NotImplementedError

class Countdown:
    # Custom iterator counting DOWN from start to 0, inclusive, then stopping.
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        # TODO: implement -- raise StopIteration once current is less than 0
        raise NotImplementedError

@contextmanager
def suppress_and_log(*exceptions):
    # Context manager: run the block; if it raises one of `exceptions`, print a message
    # and suppress it so it does not propagate. Other exception types should still propagate.
    # TODO: implement using try/except around a yield
    raise NotImplementedError

In [14]:
def _check(label, ok, detail=""):
    print("  [" + ("PASS" if ok else "FAIL") + "]", label, detail)

try:
    result = list(chunked([1, 2, 3, 4, 5, 6, 7], 3))
    _check("chunked splits into groups of 3", result == [[1, 2, 3], [4, 5, 6], [7]], result)
except NotImplementedError:
    print("  [SKIP] chunked -- not implemented yet")
except Exception as e:
    print("  [ERROR] chunked --", e)

try:
    result = list(Countdown(3))
    _check("Countdown(3) counts down to 0", result == [3, 2, 1, 0], result)
except NotImplementedError:
    print("  [SKIP] Countdown -- not implemented yet")
except Exception as e:
    print("  [ERROR] Countdown --", e)

try:
    with suppress_and_log(ValueError):
        raise ValueError("test error")
    _check("suppress_and_log suppresses the listed exception type", True)
except NotImplementedError:
    print("  [SKIP] suppress_and_log -- not implemented yet")
except Exception as e:
    print("  [ERROR] suppress_and_log --", e)

try:
    propagated = False
    try:
        with suppress_and_log(ValueError):
            raise TypeError("a different exception type")
    except TypeError:
        propagated = True
    _check("suppress_and_log lets OTHER exception types propagate", propagated)
except NotImplementedError:
    pass
except Exception as e:
    print("  [ERROR] suppress_and_log propagation check --", e)

  [SKIP] chunked -- not implemented yet
  [SKIP] Countdown -- not implemented yet
  [SKIP] suppress_and_log -- not implemented yet


## Self-check before moving on

- [ ] I can implement a custom iterator using __iter__ and __next__, and explain StopIteration
- [ ] I can explain the difference between an iterable and an iterator, and why lists survive multiple loops but an exhausted generator does not
- [ ] I can write a generator function using yield and explain why it needs far less boilerplate than a hand-written iterator class
- [ ] I can explain the memory difference between a list comprehension and a generator expression, and when that difference actually matters
- [ ] I know at least four itertools functions, such as chain, islice, groupby, product, and when to reach for each
- [ ] I can write a context manager both as a class with __enter__/__exit__ and with @contextlib.contextmanager
- [ ] I can explain why with guarantees cleanup even when the block raises an exception

Next: `05-concurrency-threading-gil-multiprocessing.ipynb`